In [43]:
import pandas as pd


df = pd.read_csv('C:/Users/Usuario/Desktop/Cd/regular_season_totals_2010_2024.csv')


df['win'] = df['WL'] == 'W'
df['loss'] = df['WL'] == 'L'

# Criar coluna pontos "tomados"
opp_pts = (
    df[['GAME_ID', 'TEAM_ID', 'PTS']]
    .rename(columns={'TEAM_ID': 'OPP_ID', 'PTS': 'PTS_TKN'})
)
df_merged = df.merge(opp_pts, on='GAME_ID')
df_merged = df_merged[df_merged['TEAM_ID'] != df_merged['OPP_ID']]


metrics = [
    'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
    'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS'
]

agg = (
    df_merged
    .groupby(['SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME'])
    .agg(
        WINS=('win', 'sum'),
        LOSSES=('loss', 'sum'),
        PTS_TKN=('PTS_TKN', 'mean'),
        **{col: (col, 'mean') for col in metrics}
    )
    .reset_index()
)

# Criar coluna rank vitórias (media pontos para desempate)
agg = agg.sort_values(
    ['SEASON_YEAR', 'WINS', 'PTS'],
    ascending=[True, False, False]
)
agg['W_RANK'] = agg.groupby('SEASON_YEAR').cumcount() + 1

cols = [
    'SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'W_RANK',
    'WINS', 'LOSSES', 'PTS', 'PTS_TKN', 'PLUS_MINUS'
] + [c for c in agg.columns if c not in {
    'SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'W_RANK',
    'WINS', 'LOSSES', 'PTS', 'PTS_TKN', 'PLUS_MINUS'
}]

agg = agg[cols]

print(agg.head())

   SEASON_YEAR     TEAM_ID TEAM_ABBREVIATION           TEAM_NAME  W_RANK   
4      2010-11  1610612741               CHI       Chicago Bulls       1  \
22     2010-11  1610612759               SAS   San Antonio Spurs       2   
11     2010-11  1610612748               MIA          Miami Heat       3   
10     2010-11  1610612747               LAL  Los Angeles Lakers       4   
5      2010-11  1610612742               DAL    Dallas Mavericks       5   

    WINS  LOSSES         PTS    PTS_TKN  PLUS_MINUS  ...       OREB   
4     62      20   98.621951  91.304878    7.317073  ...  11.792683  \
22    61      21  103.682927  97.975610    5.707317  ...  10.109756   
11    58      24  102.060976  94.597561    7.463415  ...   9.634146   
10    57      25  101.475610  95.365854    6.109756  ...  12.060976   
5     57      25  100.243902  96.012195    4.231707  ...   9.512195   

         DREB        REB        AST        TOV       STL       BLK      BLKA   
4   32.365854  44.158537  22.280488 

In [45]:
agg.to_csv('C:/Users/Usuario/Desktop/Cd/stats_by_team_by_season.csv', index=False, float_format='%.2f')

In [42]:
# 1) Count how many teams have each win‐total per season
win_counts = (
    agg
    .groupby(['SEASON_YEAR', 'WINS'])
    .size()
    .reset_index(name='teams_with_that_win_total')
)

# 2) Filter to only those (season, wins) combos where more than one team shares the same wins
ties = win_counts[win_counts['teams_with_that_win_total'] > 1]

print("Seasons with tied win totals:")
print(ties)

# Mark every row in agg where its (season, wins) occurs >1 times
agg['is_tied'] = (
    agg
    .groupby(['SEASON_YEAR', 'WINS'])['WINS']
    .transform('count') > 1
)

# Extract just those tied teams
tied_teams = agg[agg['is_tied']]

print("Teams that are tied on wins within their season:")
print(tied_teams[['SEASON_YEAR','TEAM_NAME','WINS', 'W_RANK']])

Seasons with tied win totals:
    SEASON_YEAR  WINS  teams_with_that_win_total
4       2010-11    24                          2
17      2010-11    46                          2
23      2010-11    57                          2
29      2011-12    21                          2
30      2011-12    22                          2
..          ...   ...                        ...
304     2023-24    46                          3
305     2023-24    47                          4
307     2023-24    49                          3
308     2023-24    50                          2
311     2023-24    57                          2

[80 rows x 3 columns]
Teams that are tied on wins within their season:
    SEASON_YEAR               TEAM_NAME  WINS  W_RANK
10      2010-11      Los Angeles Lakers    57       4
5       2010-11        Dallas Mavericks    57       5
26      2010-11       Memphis Grizzlies    46      11
3       2010-11     New Orleans Hornets    46      12
21      2010-11        Sacramento Kings 

In [36]:
# 1) Count how many teams have each win‐total per season
win_counts = (
    agg
    .groupby(['SEASON_YEAR', 'WINS'])
    .size()
    .reset_index(name='teams_with_that_win_total')
)

# 2) Filter to only those (season, wins) combos where more than one team shares the same wins
ties = win_counts[win_counts['teams_with_that_win_total'] > 1]

print("Seasons with tied win totals:")
print(ties)

# Mark every row in agg where its (season, wins) occurs >1 times
agg['is_tied'] = (
    agg
    .groupby(['SEASON_YEAR', 'WINS'])['WINS']
    .transform('count') > 1
)

# Extract just those tied teams
tied_teams = agg[agg['is_tied']]

print("Teams that are tied on wins within their season:")
print(tied_teams[['SEASON_YEAR','TEAM_NAME','WINS', 'W_RANK']])


Seasons with tied win totals:
    SEASON_YEAR  WINS  teams_with_that_win_total
4       2010-11    24                          2
17      2010-11    46                          2
23      2010-11    57                          2
29      2011-12    21                          2
30      2011-12    22                          2
..          ...   ...                        ...
304     2023-24    46                          3
305     2023-24    47                          4
307     2023-24    49                          3
308     2023-24    50                          2
311     2023-24    57                          2

[80 rows x 3 columns]
Teams that are tied on wins within their season:
    SEASON_YEAR               TEAM_NAME  WINS  W_RANK
3       2010-11     New Orleans Hornets    46      11
5       2010-11        Dallas Mavericks    57       4
10      2010-11      Los Angeles Lakers    57       4
14      2010-11         New Jersey Nets    24      25
21      2010-11        Sacramento Kings 